In [1]:
from __future__ import annotations

import os
# os.environ["HWLOC_HIDE_ERRORS"] = "1"
# os.environ["TF_ENABLE_NUMA"] = "0"
# os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
# os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".80"
# 3. Provide valid temporary directory for ptxas CUDA kernel compilation
os.environ["TMPDIR"] = "/tmp"
os.environ["TMP"] = "/tmp"
os.environ["TEMP"] = "/tmp"

import jax.numpy as jnp
import s2fft
import argparse
import shutil
from pathlib import Path

import numpy as np
import jax
jax.config.update("jax_enable_x64", True)
print("Jax version", jax.__version__, "64-bit", jax.config.read("jax_enable_x64"))
print("jax", jax.__version__, "GPU backend:", jax.default_backend() == 'gpu', f"({jax.devices('gpu')[0].device_kind})" if jax.default_backend() == 'gpu' else "")
import jax.numpy as jnp
import s2fft
from tqdm.auto import tqdm
from typing import Literal


def resample_mw_to_mwss(map_path: Path, output_path: Path) -> None:
    m = np.load(map_path)
    m = m.reshape((1500, 2999, 5)).astype(np.float32)

    m_mwss = np.stack(
        [
            np.array(s2fft.utils.resampling.mw_to_mwss(m[..., c], L=1500)).real
            for c in range(5)
        ],
        axis=-1,
    )

    np.save(output_path, m_mwss.astype(np.float64, copy=False))

def L_downsample_mwss(map_path: Path, output_path: Path, L_in: int, L_out: int) -> None:
    m = np.load(map_path)
    # m_halfL = np.stack(
    #         [
    #             np.array(s2fft.utils.resampling.downsample_by_two_mwss(m[..., c], L=L_in)).real
    #             for c in range(5)
    #         ], 
    #     axis=-1
    #     )  # shape: (L_in/2, ...)
    # m_halfL = np.array(s2fft.utils.resampling.downsample_by_two_mwss(m, L=L_in)).real
    # m = m[0:-1:2, 0:-1:2, :]
    def L_downsample(f_map, L_in, L_out):
        L_cut = L_in-L_out
        m = s2fft.transforms.spherical.forward(f_map, L=L_in, reality=True, method='jax_cuda', sampling='mwss')
        m = m[:L_out, L_cut:-L_cut]  # keep modes l < 750
        m = s2fft.transforms.spherical.inverse(m, L=L_out, reality=True, method='jax_cuda', sampling='mwss')
        return m
    m = np.stack(
            [
                np.array(L_downsample(m[..., c], L_in=L_in, L_out=L_out)).real
                for c in range(5)
            ], 
        axis=-1
        )  # shape: (L_in/2, ...)

    np.save(output_path, m.astype(np.float64, copy=False))
    del m
    # raise NotImplementedError("L_downsample_by_two_mwss is not implemented yet.")



JAX is not using 64-bit precision. This will dramatically affect numerical precision at even moderate L.


Jax version 0.11.0 64-bit True
jax 0.11.0 GPU backend: True (NVIDIA GH200 120GB)


E0803 14:11:18.858421   83248 numa_hwloc.cc:121] Call to hwloc_set_cpubind() failed: Invalid argument [22]


In [ ]:
SOURCE_DIR = Path(
    "/projects/u6pf/brianycc/spherical_maps/"
    "5_tomo_bins_flm_1500_batch_1_mwss/"
)
OUTPUT_DIR = Path(
    "/projects/u6pf/brianycc/spherical_maps/"
    "5_tomo_bins_flm_750_batch_1_mwss/"
)
OVERWRITE = False


def main() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    params_src = SOURCE_DIR / "params.npy"
    params_dst = OUTPUT_DIR / "params.npy"
    if params_src.exists() and (OVERWRITE or not params_dst.exists()):
        shutil.copy2(params_src, params_dst)

    map_files = sorted(SOURCE_DIR.glob("map_*.npy"))
    if not map_files:
        raise FileNotFoundError(f"No map_*.npy files found in {SOURCE_DIR}")
``
    converted = 0
    skipped = 0

    for map_path in tqdm(map_files, desc="Resampling maps"):
        output_path = OUTPUT_DIR / map_path.name
        if output_path.exists() and not OVERWRITE:
            skipped += 1
            continue

        # resample_mw_to_mwss(map_path, output_path)
        L_downsample_mwss(map_path, output_path, L_in=1500, L_out=750)
        converted += 1

    print(
        f"Converted {converted} maps, skipped {skipped} existing maps.\n"
        f"Source: {SOURCE_DIR}\n"
        f"Output: {OUTPUT_DIR}"
    )

print("SOURCE_DIR =", SOURCE_DIR)
print("Exists     =", SOURCE_DIR.exists())
print("Is dir     =", SOURCE_DIR.is_dir())
map_files = sorted(SOURCE_DIR.glob("map_*.npy"))
print("Count      =", len(map_files))
print("First 5    =", [p.name for p in map_files[:5]])

main()

SOURCE_DIR = /projects/u6pf/brianycc/spherical_maps/5_tomo_bins_flm_1500_batch_1_mwss
Exists     = True
Is dir     = True
Count      = 2000
First 5    = ['map_0.npy', 'map_1.npy', 'map_10.npy', 'map_100.npy', 'map_1000.npy']


Resampling maps:   0%|                                                                                                               | 0/2000 [00:00<?, ?it/s]

Resampling maps: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 2000/2000 [2:21:54<00:00,  4.26s/it]

Converted 2000 maps, skipped 0 existing maps.
Source: /projects/u6pf/brianycc/spherical_maps/5_tomo_bins_flm_1500_batch_1_mwss
Output: /projects/u6pf/brianycc/spherical_maps/5_tomo_bins_flm_750_batch_1_mwss


In [29]:
check_path = SOURCE_DIR / "map_0.npy"
m = np.load(check_path)
print("Check path:", check_path)
print("Shape:", m.shape)

check_path = OUTPUT_DIR / "map_0.npy"
m = np.load(check_path)
print("Check path:", check_path)
print("Shape:", m.shape)

Check path: /projects/u6pf/brianycc/spherical_maps/5_tomo_bins_flm_1500_batch_1_mwss/map_0.npy
Shape: (1501, 3000, 5)
Check path: /projects/u6pf/brianycc/spherical_maps/5_tomo_bins_flm_750_batch_1_mwss/map_0.npy
Shape: (750, 1500, 5)


In [2]:
import os
import glob
import numpy as np
from tqdm import tqdm

def compute_and_save_map_stats(directory, output_filename="map_stats.npz"):
    """
    Computes channel-wise mean and std across all map_{i}.npy files in `directory`
    loading only one map into memory at a time. Saves stats into `output_filename`.
    """
    map_files = glob.glob(os.path.join(directory, "map_*.npy"))
    n_files = len(map_files)
    assert n_files > 0, f"No map_*.npy files found in {directory}"

    print(f"Found {n_files} map files. Calculating mean and std...")

    # Inspect shape of first map to initialize accumulators
    sample_map = np.load(map_files[0])
    num_channels = sample_map.shape[-1]
    
    # Online accumulators for each channel
    count = 0
    mean = np.zeros(num_channels, dtype=np.float64)
    M2 = np.zeros(num_channels, dtype=np.float64)

    # Welford's algorithm over each map loading only 1 map at a time
    for map_path in tqdm(map_files):
        m = np.load(map_path).astype(np.float64)
        # Collapse spatial dimensions (theta, phi) -> reshape to (-1, channels)
        pixels = m.reshape(-1, num_channels)
        
        n_pixels = pixels.shape[0]
        batch_mean = np.mean(pixels, axis=0)
        batch_var = np.var(pixels, axis=0)

        # Update global accumulators (Welford / Parallel algorithm)
        if count == 0:
            mean = batch_mean
            M2 = batch_var * n_pixels
            count = n_pixels
        else:
            delta = batch_mean - mean
            new_count = count + n_pixels
            mean += delta * n_pixels / new_count
            M2 += batch_var * n_pixels + (delta ** 2) * count * n_pixels / new_count
            count = new_count

    variance = M2 / count
    std = np.sqrt(variance)

    # Cast stats back to float32 for model training
    mean = mean.astype(np.float32)
    std = std.astype(np.float32)

    stats_path = os.path.join(directory, output_filename)
    np.savez(stats_path, mean=mean, std=std)
    print(f"Successfully saved stats to {stats_path}")
    print("Channel Mean:", mean)
    print("Channel Std :", std)
    return stats_path

# Execute calculation for your map directory
data_dir = "/projects/u6pf/brianycc/spherical_maps/5_tomo_bins_flm_750_batch_1_mwss/"
compute_and_save_map_stats(data_dir)


Found 2000 map files. Calculating mean and std...


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2000/2000 [01:49<00:00, 18.19it/s]

Successfully saved stats to /projects/u6pf/brianycc/spherical_maps/5_tomo_bins_flm_750_batch_1_mwss/map_stats.npz
Channel Mean: [0.00050952 0.00050918 0.00050899 0.00050917 0.00050933]
Channel Std : [0.00812444 0.00812473 0.0081247  0.00812403 0.00812417]


'/projects/u6pf/brianycc/spherical_maps/5_tomo_bins_flm_750_batch_1_mwss/map_stats.npz'

[0 1 2 3 4 5 6 7 8 9]
[7 6 5 4]


In [ ]:
# def main() -> None:
#     parser = argparse.ArgumentParser(
#         description="Bulk-resample spherical maps from MW to MWSS."
#     )
#     parser.add_argument(
#         "--source-dir",
#         type=Path,
#         default=Path(
#             "/projects/u6pf/brianycc/spherical_maps/"
#             "map_compression_samples_5_tomo_bins_flm_1500_batch_1"
#         ),
#         help="Directory containing map_*.npy and params.npy",
#     )
#     parser.add_argument(
#         "--output-dir",
#         type=Path,
#         default=Path(
#             "/projects/u6pf/brianycc/spherical_maps/"
#             "map_compression_samples_5_tomo_bins_flm_1500_batch_1_mwss"
#         ),
#         help="Directory where MWSS-resampled files will be written",
#     )
#     args = parser.parse_args()

#     source_dir: Path = args.source_dir
#     output_dir: Path = args.output_dir
#     output_dir.mkdir(parents=True, exist_ok=True)

#     params_src = source_dir / "params.npy"
#     params_dst = output_dir / "params.npy"
#     if params_src.exists() and not params_dst.exists():
#         shutil.copy2(params_src, params_dst)

#     map_files = sorted(source_dir.glob("map_*.npy"))
#     if not map_files:
#         raise FileNotFoundError(f"No map_*.npy files found in {source_dir}")

#     skipped = 0
#     converted = 0

#     for map_path in map_files:
#         output_path = output_dir / map_path.name
#         if output_path.exists():
#             skipped += 1
#             continue
#         resample_mw_to_mwss(map_path, output_path)
#         converted += 1

#     print(
#         f"Converted {converted} maps, skipped {skipped} existing maps "
#         f"from {source_dir} to {output_dir}"
#     )


# if __name__ == "__main__":
#     main()